In [3]:
import os
# 缓解显存碎片和过度预留的问题
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [7]:
import json
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, GenerationConfig
from tqdm import tqdm
import openai
import json
import time
import os
from PIL import Image
import re

In [5]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"
ocr_model_path = "../model/deepseek-ocr"

tokenizer = AutoTokenizer.from_pretrained(ocr_model_path, _attn_implementation='flash_attention_2', trust_remote_code=True)
model = AutoModel.from_pretrained(
    ocr_model_path, trust_remote_code=True, use_safetensors=True
)
model = model.eval().cuda("cuda:1").to(torch.bfloat16)

# image_file = 'your_image.jpg'
# output_path = 'your/output/dir'

# infer(self, tokenizer, prompt='', image_file='', output_path = ' ', base_size = 1024, image_size = 640, crop_mode = True, test_compress = False, save_results = False):

# Tiny: base_size = 512, image_size = 512, crop_mode = False
# Small: base_size = 640, image_size = 640, crop_mode = False
# Base: base_size = 1024, image_size = 1024, crop_mode = False
# Large: base_size = 1280, image_size = 1280, crop_mode = False

# Gundam: base_size = 1024, image_size = 640, crop_mode = True

# res = model.infer(tokenizer, prompt=prompt, image_file=image_file, output_path = output_path, base_size = 1024, image_size = 640, crop_mode=True, save_results = True, test_compress = True)

You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.
Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at ../model/deepseek-ocr and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
# 论文中使用的prompt
prompt = "<image>\nFree OCR. "
# 官方仓库实例的prompt
# prompt = "<image>\n<|grounding|>Convert the document to markdown. "  

In [11]:
def load_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# 原论文中只评估了 tiny 和 small 模型 分别对应的image_size base_size = 512 640
def process_single_image(tokenizer, model, image_name, output_path, imgs_dir, mode, prompt):
    """
    处理单张图片，返回清理后的 OCR 文本
    """
    if mode == "tiny":
        IMAGE_SIZE = 512
        BASE_SIZE = 512
    elif mode == "small":
        IMAGE_SIZE = 640
        BASE_SIZE = 640
    elif mode == "raw":
        img = Image.open(os.path.join(imgs_dir, image_name))
        w, h = img.size
        long_side = max(w, h)
        IMAGE_SIZE = long_side
        BASE_SIZE = long_side
        
    image_path = os.path.join(imgs_dir, image_name)
    
    if mode == "raw":
        # 不使用压缩的 OCR 结果, 作为对比实验
        test_compress = False
    else:
        test_compress = True
        
    res = model.infer(
        tokenizer=tokenizer,
        prompt=prompt,
        image_file=image_path,
        output_path=output_path,
        base_size=BASE_SIZE,
        image_size=IMAGE_SIZE,
        # crop_mode=True,
        crop_mode=False,
        # save_results=True,    # 这个设置会将结果保存到output_path目录下
        save_results=False,
        eval_mode=True,         # 评估模式，不保存结果，将结果返回
        test_compress=test_compress,     # 使用压缩的 OCR 结果
    )
    clean_text = clean_ocr_output(res)
    return clean_text

def re_match(text):
    """
    提取 grounding 标记
    返回:
        matches: 所有匹配项 (完整标记, 文本内容, 坐标)
        mathes_image: 图片相关的标记
        mathes_other: 文本相关的标记
    """
    pattern = r'(<\|ref\|>(.*?)<\|/ref\|><\|det\|>(.*?)<\|/det\|>)'
    matches = re.findall(pattern, text, re.DOTALL)

    mathes_image = []
    mathes_other = []
    for a_match in matches:
        if '<|ref|>image<|/ref|>' in a_match[0]:
            mathes_image.append(a_match[0])
        else:
            mathes_other.append(a_match[0])
    return matches, mathes_image, mathes_other

def clean_ocr_output(text):
    """
    官方的清理方法：
    1. 提取所有标记
    2. 替换图片标记为 markdown 图片格式
    3. 删除所有文本标记
    """
    matches_ref, matches_images, mathes_other = re_match(text)
    
    # 替换图片标记
    for idx, a_match_image in enumerate(matches_images):
        text = text.replace(a_match_image, f'![](images/{idx}.jpg)\n')
    
    # 删除所有文本标记
    for idx, a_match_other in enumerate(mathes_other):
        text = text.replace(a_match_other, '')
    
    # 额外清理
    text = text.replace('\\coloneqq', ':=').replace('\\eqqcolon', '=:')
    text = text.replace('\n\n\n\n', '\n\n').replace('\n\n\n', '\n\n')
    text = text.replace('<center>', '').replace('</center>', '')
    
    return text.strip()

def ocr(tokenizer, model, data_path=None, output_path = "../output", save_path=None, imgs_dir=None, mode="tiny"):
    ocr_results = []
    image_names = [f"en_{i+1}.png" for i in range(112)]
    data = load_data(data_path)
    
    image_names = image_names[:4]
    data = data[:4]
    
    # image_paths = [os.path.join(images_dir, img_name) for img_name in image_names]
    print(f"开始处理 {len(image_names)} 张图片...")
    
    for image_name in tqdm(image_names):
        clean_text = process_single_image(tokenizer, model, image_name, output_path, imgs_dir, mode, prompt)
        ocr_results.append({
            "image": image_name,
            "ocr_text": clean_text
        })
    # 按照image name重新排序
    ocr_results = sorted(
        ocr_results,
        key=lambda x: int(x["image"].split("_")[-1].split(".")[0])
    )
    
    # 将结果合并到原始数据中
    for item in data:
        image_name = item["image"]
        for ocr_item in ocr_results:
            if ocr_item["image"] == image_name:
                item["ocr_text"] = ocr_item["ocr_text"]
                break
    
    # 将最终结果保存到文件中
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    
    print(f"\n结果已保存到: {save_path}")

## OCR

In [13]:
ocr(
    tokenizer,
    model,
    data_path="../fox_data/replace.json",
    output_path="../output",
    save_path="../results/replace/en_png_tiny.json",
    imgs_dir="../fox_data/replace/",
    mode="tiny"
)

开始处理 4 张图片...


  0%|          | 0/4 [00:00<?, ?it/s]

directly resize


/root/miniconda3/envs/py311/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 64, 1280])
NO PATCHES


...

directly resize


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 64, 1280])
NO PATCHES


100%|██████████| 4/4 [03:17<00:00, 49.28s/it]


结果已保存到: ../results/replace/en_png_tiny.json


In [14]:
ocr(
    tokenizer,
    model,
    data_path="../fox_data/data.json",
    output_path="../output",
    save_path="../results/replace/en_png_tiny_no.json",
    imgs_dir="../fox_data/en_png",
    mode="tiny"
)

开始处理 4 张图片...


  0%|          | 0/4 [00:00<?, ?it/s]

directly resize


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 64, 1280])
NO PATCHES


...

directly resize


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 64, 1280])
NO PATCHES


100%|██████████| 4/4 [03:27<00:00, 51.99s/it]


结果已保存到: ../results/replace/en_png_tiny_no.json
